# Phase 09 — Clean-v3 KUB baseline training

Reconstructed reproducibility notebook for the `phase_09_clean_v3_baseline` directory, which contained no notebook. All four KUB single-domain baselines were trained under the clean-v3 protocol and their per-epoch histories survive.

> **Status:** reconstructed analysis notebook. It reads persisted artifacts only. No training, no new inference, no protected TEST access.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIG = REPORT / "figures"

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

def fig(subdir, name, width=760):
    """Display a generated figure, failing loudly if it is missing."""
    p = FIG / subdir / f"{name}.png"
    if not p.exists():
        raise FileNotFoundError(f"missing figure: {p}")
    display(Image(filename=str(p), width=width))

def table(path, **kw):
    return pd.read_csv(path, **kw)

GUARD.require()

True

In [2]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified


In [3]:
hist = pd.read_csv(REPO / "reports/tables/phase10_final_all_models_epoch_results_long.csv")
print("models:", hist.model.nunique(), "| rows:", len(hist))
print(hist.groupby("model").epoch.max().to_string())

models: 4 | rows: 280
model
Attention U-Net    69
DeepLabV3+         67
UNet               59
UNet++             85


In [4]:
# Per-epoch validation Dice for the four KUB baselines.
import matplotlib.pyplot as plt
fig_, ax = plt.subplots(figsize=(6.4, 3.0))
for name, g in hist.groupby("model"):
    g = g.sort_values("epoch")
    ax.plot(g.epoch, g.val_macro_dice, linewidth=1.2, label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Dice (KUB)")
ax.set_title("Phase 09/10 clean-v3 KUB baselines — genuine per-epoch history")
ax.legend(fontsize=6.5); plt.tight_layout(); plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_40096\2303979039.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=6.5); plt.tight_layout(); plt.show()


In [5]:
# Best epoch and final metric per model, taken from the recorded flags.
rows = []
for name, g in hist.groupby("model"):
    g = g.sort_values("epoch")
    best = g[g.is_best_epoch.astype(str).str.lower() == "true"]
    rows.append({"model": name,
                 "epochs_completed": int(g.epoch.max()),
                 "best_epoch": int(best.epoch.iloc[0]) if len(best) else None,
                 "best_val_dice": round(float(g.val_macro_dice.max()), 6),
                 "final_val_dice": round(float(g.val_macro_dice.iloc[-1]), 6)})
pd.DataFrame(rows)

,model,epochs_completed,best_epoch,best_val_dice,final_val_dice
0,Attention U-Net,69,54,0.752513,0.744384
1,DeepLabV3+,67,52,0.750914,0.742612
2,UNet,59,44,0.740613,0.737066
3,UNet++,85,70,0.764160,0.747758


In [6]:
# Loss trajectories.
fig_, ax = plt.subplots(figsize=(6.4, 2.8))
for name, g in hist.groupby("model"):
    g = g.sort_values("epoch")
    ax.plot(g.epoch, g.train_loss, linewidth=1.0, linestyle="-", label=f"{name} train")
    ax.plot(g.epoch, g.val_loss, linewidth=1.0, linestyle="--", label=f"{name} val")
ax.set_xlabel("Epoch"); ax.set_ylabel("BCE + Dice loss")
ax.set_title("Training and validation loss (solid / dashed)")
ax.legend(fontsize=5.4, ncol=2); plt.tight_layout(); plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_40096\4124432287.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=5.4, ncol=2); plt.tight_layout(); plt.show()


In [7]:
# Learning-rate schedule actually used.
g = hist[hist.model == "UNet++"].sort_values("epoch")
fig_, ax = plt.subplots(figsize=(5.6, 2.2))
ax.plot(g.epoch, g.learning_rate, color="#CC79A7", linewidth=1.3)
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate")
ax.set_title("Cosine annealing, recorded per epoch (UNet++)")
plt.tight_layout(); plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_40096\198713138.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Provenance

- `reports/tables/phase10_final_all_models_epoch_results_long.csv`
- `experiments/phase9_kub_unet_baseline_seed42_512_clean_v3/history.csv`
- `experiments/phase10_kub_*/history.csv`

In [8]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified
